In [ ]:
# Tokenek

from selenium import webdriver
import json
import time
driver = webdriver.Chrome()

url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/"
driver.get(url)

print("Oldal betöltve, hálózati forgalom figyelése...")
time.sleep(12)  # várjuk meg a GraphQL hívásokat

logs = driver.get_log("performance")

umbrella_token = None
visitor_id = None
rand_id = None

for entry in logs:
    message = json.loads(entry["message"])["message"]

    if (
        message["method"] == "Network.requestWillBeSent"
        and "graphql" in message["params"]["request"]["url"]
    ):
        headers = message["params"]["request"]["headers"]

        umbrella_token = headers.get("kw-umbrella-token")
        visitor_id = headers.get("kw-skypicker-visitor-uniqid")
        rand_id = headers.get("kw-x-rand-id")

        if umbrella_token or visitor_id:
            print("\n🎯 GraphQL request elkapva!")
            break

driver.quit()

print("\n===== TOKENEK =====")
print("kw-umbrella-token:", umbrella_token)
print("kw-skypicker-visitor-uniqid:", visitor_id)
print("kw-x-rand-id:", rand_id)


In [24]:
# Repjegyek proto

from selenium import webdriver
import json
import time
import requests

# 1. Tokenek megszerzése Seleniummal
print("🚀 Tokenek megszerzése...")
driver = webdriver.Chrome()

url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/"
driver.get(url)

print("⏳ Oldal betöltve, várakozás a GraphQL hívásokra...")
time.sleep(6)

logs = driver.get_log("performance")

umbrella_token = None
visitor_id = None
rand_id = None

for entry in logs:
    message = json.loads(entry["message"])["message"]
    
    if (
        message["method"] == "Network.requestWillBeSent"
        and "graphql" in message["params"]["request"]["url"]
    ):
        headers = message["params"]["request"]["headers"]
        
        umbrella_token = headers.get("kw-umbrella-token")
        visitor_id = headers.get("kw-skypicker-visitor-uniqid")
        rand_id = headers.get("kw-x-rand-id")
        
        if umbrella_token:
            break

driver.quit()

print("\n✅ Tokenek megvannak:")
print(f"  umbrella: {umbrella_token[:50]}..." if umbrella_token else "  umbrella: None")
print(f"  visitor: {visitor_id}")
print(f"  rand_id: {rand_id}")

# 2. GraphQL query - JAVÍTOTT verzió az ItineraryReturn típussal
print("\n🔍 Járatok lekérése...")

graphql_payload = {
    "query": """
    query SearchReturnItinerariesQuery(
      $search: SearchReturnInput
      $filter: ItinerariesFilterInput
      $options: ItinerariesOptionsInput
    ) {
      returnItineraries(search: $search, filter: $filter, options: $options) {
        __typename
        ... on Itineraries {
          metadata {
            itinerariesCount
          }
          itineraries {
            __typename
            ... on ItineraryReturn {
              id
              legacyId
              price {
                amount
              }
              priceEur {
                amount
              }
              duration
              provider {
                name
                code
              }
              outbound {
                id
                duration
                sectorSegments {
                  segment {
                    id
                    source {
                      localTime
                      utcTimeIso
                      station {
                        code
                        name
                        city {
                          name
                        }
                      }
                    }
                    destination {
                      localTime
                      utcTimeIso
                      station {
                        code
                        name
                        city {
                          name
                        }
                      }
                    }
                    carrier {
                      name
                      code
                    }
                    duration
                  }
                }
              }
              inbound {
                id
                duration
                sectorSegments {
                  segment {
                    id
                    source {
                      localTime
                      station {
                        code
                        name
                      }
                    }
                    destination {
                      localTime
                      station {
                        code
                        name
                      }
                    }
                    carrier {
                      name
                      code
                    }
                  }
                }
              }
            }
          }
        }
        ... on AppError {
          error: message
        }
      }
    }
    """,
    "variables": {
        "search": {
            "itinerary": {
                "source": {"ids": ["City:budapest_hu"]},
                "destination": {"ids": ["City:barcelona_es"]}
            },
            "passengers": {
                "adults": 1
            }
        },
        "filter": {
            "transportTypes": ["FLIGHT"],
            "limit": 10
        },
        "options": {
            "currency": "huf",
            "locale": "hu",
            "market": "hu",
            "partner": "skypicker"
        }
    }
}

# Requests használata
headers = {
    "Content-Type": "application/json",
    "kw-umbrella-token": umbrella_token,
    "kw-skypicker-visitor-uniqid": visitor_id,
    "kw-x-rand-id": rand_id,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.post(
    "https://api.skypicker.com/umbrella/v2/graphql?featureName=SearchReturnItinerariesQuery",
    json=graphql_payload,
    headers=headers,
    timeout=30
)

# Válasz feldolgozása
data = response.json()

print(f"\n✅ Válasz érkezett (status: {response.status_code})")

# Hibakezelés
if "errors" in data:
    print("\n❌ GraphQL hibák:")
    for error in data["errors"]:
        print(f"  - {error['message']}")
    print("\n💾 Hibás válasz mentve: kiwi_error.json")
    with open("kiwi_error.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
elif data.get("data") and data["data"].get("returnItineraries"):
    result = data["data"]["returnItineraries"]
    
    if result["__typename"] == "Itineraries":
        itineraries = result.get("itineraries", [])
        
        print(f"\n🎫 Talált járatok száma: {len(itineraries)}\n")
        
        for i, flight in enumerate(itineraries[:5], 1):
            if flight["__typename"] != "ItineraryReturn":
                continue
                
            price = float(flight["price"]["amount"])
            duration = flight["duration"] // 3600  # óra
            provider = flight["provider"]["name"]
            
            # Outbound (oda)
            outbound = flight["outbound"]["sectorSegments"][0]["segment"]
            dep = outbound["source"]["station"]["code"]
            dep_city = outbound["source"]["station"]["city"]["name"]
            arr = outbound["destination"]["station"]["code"]
            arr_city = outbound["destination"]["station"]["city"]["name"]
            dep_time = outbound["source"]["localTime"]
            carrier = outbound["carrier"]["name"]
            
            # Inbound (vissza)
            inbound = flight["inbound"]["sectorSegments"][0]["segment"]
            ret_time = inbound["source"]["localTime"]
            
            print(f"{i}. {dep_city} ({dep}) ⇄ {arr_city} ({arr})")
            print(f"   Ár: {price:,.0f} HUF | Időtartam: ~{duration}h")
            print(f"   Oda: {dep_time} | Vissza: {ret_time}")
            print(f"   Légitársaság: {carrier} | {provider}")
            print()
    else:
        print(f"\n❌ Hiba: {result.get('error', 'Ismeretlen hiba')}")
else:
    print("\n❌ Nem érkezett adat")

🚀 Tokenek megszerzése...
⏳ Oldal betöltve, várakozás a GraphQL hívásokra...

✅ Tokenek megvannak:
  umbrella: 127023f99e6012d1485bc942f7e4ab6cd3c2727040a9d8e3ef...
  visitor: b07e15c2-b170-4340-9666-ffb0670de46e
  rand_id: 58fe098a3b2f66b7b435a6ed150efcba9766811c

🔍 Járatok lekérése...

✅ Válasz érkezett (status: 200)

🎫 Talált járatok száma: 10

1. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 20,109 HUF | Időtartam: ~5h
   Oda: 2026-01-23T17:45:00 | Vissza: 2026-02-04T18:50:00
   Légitársaság: Ryanair | Kiwi.com

2. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,252 HUF | Időtartam: ~5h
   Oda: 2026-02-25T15:35:00 | Vissza: 2026-03-09T06:25:00
   Légitársaság: Ryanair | Kiwi.com

3. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,252 HUF | Időtartam: ~5h
   Oda: 2026-02-25T15:35:00 | Vissza: 2026-03-11T09:30:00
   Légitársaság: Ryanair | Kiwi.com

4. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,668 HUF | Időtartam: ~5h
   Oda: 2026-01-23T17:45:00 | Vissza: 2026-02-01T21:00:00
   Légitársaság: Ryanair

In [11]:
from selenium import webdriver
import json
import time
import requests
import pandas as pd
from datetime import datetime
from typing import Optional, List
from itertools import product

def get_kiwi_tokens(headless: bool = False) -> dict:
    """
    Kiwi.com tokenek megszerzése Selenium segítségével.
    
    Args:
        headless: Ha True, háttérben fut a böngésző
        
    Returns:
        Dictionary a tokenekkel: umbrella_token, visitor_id, rand_id
    """
    print("🚀 Tokenek megszerzése...")
    
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless')
    
    # FONTOS: Performance logging engedélyezése
    options.set_capability('goog:loggingPrefs', {'performance': 'ALL'})
    
    driver = webdriver.Chrome(options=options)
    
    # Egyszerű one-way keresés a tokenekhez
    url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/anytime/no-return/"
    driver.get(url)
    
    print("⏳ Várakozás a GraphQL hívásokra...")
    time.sleep(12)
    
    logs = driver.get_log("performance")
    
    umbrella_token = None
    visitor_id = None
    rand_id = None
    
    for entry in logs:
        message = json.loads(entry["message"])["message"]
        
        if (
            message["method"] == "Network.requestWillBeSent"
            and "graphql" in message["params"]["request"]["url"]
        ):
            headers = message["params"]["request"]["headers"]
            
            umbrella_token = headers.get("kw-umbrella-token")
            visitor_id = headers.get("kw-skypicker-visitor-uniqid")
            rand_id = headers.get("kw-x-rand-id")
            
            if umbrella_token:
                break
    
    driver.quit()
    
    print("✅ Tokenek megszerzve\n")
    
    return {
        "umbrella_token": umbrella_token,
        "visitor_id": visitor_id,
        "rand_id": rand_id
    }


def search_one_way_flights(
    origin: str,
    destination: str,
    tokens: dict,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
    adults: int = 1,
    children: int = 0,
    infants: int = 0,
    limit: int = 100,
    currency: str = "huf",
    locale: str = "hu",
    max_stopovers: Optional[int] = None,
    direct_flights_only: bool = False
) -> pd.DataFrame:
    """
    Kiwi.com egyirányú járatok keresése.
    
    Args:
        origin: Indulási város/reptér kódja (pl. "budapest_hu", "BUD")
        destination: Célállomás kódja (pl. "barcelona_es", "BCN")
        tokens: Token dict a get_kiwi_tokens()-ból
        date_from: Indulás kezdő dátuma (YYYY-MM-DD), None = anytime
        date_to: Indulás záró dátuma (YYYY-MM-DD), None = anytime
        adults: Felnőttek száma
        children: Gyerekek száma
        infants: Csecsemők száma
        limit: Max találatok száma
        currency: Pénznem (huf, eur, usd)
        locale: Nyelv (hu, en)
        max_stopovers: Max átszállások száma (None = bármi)
        direct_flights_only: Csak direkt járatok
        
    Returns:
        Pandas DataFrame a találatokkal
    """
    print(f"🔍 Egyirányú járatok: {origin} → {destination}", end="")
    if date_from and date_to:
        print(f" ({date_from} - {date_to})")
    elif date_from:
        print(f" (from {date_from})")
    elif date_to:
        print(f" (until {date_to})")
    else:
        print(" (anytime)")
    
    # City ID formázás
    if ":" not in origin:
        origin_id = f"City:{origin.lower()}"
    else:
        origin_id = origin
        
    if ":" not in destination:
        dest_id = f"City:{destination.lower()}"
    else:
        dest_id = destination
    
    # GraphQL query - ONE WAY verzió
    graphql_payload = {
        "query": """
        query SearchOneWayItinerariesQuery(
          $search: SearchOnewayInput
          $filter: ItinerariesFilterInput
          $options: ItinerariesOptionsInput
        ) {
          onewayItineraries(search: $search, filter: $filter, options: $options) {
            __typename
            ... on Itineraries {
              metadata {
                itinerariesCount
              }
              itineraries {
                __typename
                ... on ItineraryOneWay {
                  id
                  shareId
                  price {
                    amount
                  }
                  priceEur {
                    amount
                  }
                  provider {
                    name
                    code
                  }
                  sector {
                    id
                    duration
                    sectorSegments {
                      segment {
                        id
                        source {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                            country {
                              code
                            }
                          }
                        }
                        destination {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                            country {
                              code
                            }
                          }
                        }
                        carrier {
                          name
                          code
                        }
                        operatingCarrier {
                          name
                          code
                        }
                        duration
                        code
                      }
                      layover {
                        duration
                      }
                    }
                  }
                  bookingOptions {
                    edges {
                      node {
                        bookingUrl
                        price {
                          amount
                        }
                      }
                    }
                  }
                }
              }
            }
            ... on AppError {
              error: message
            }
          }
        }
        """,
        "variables": {
            "search": {
                "itinerary": {
                    "source": {"ids": [origin_id]},
                    "destination": {"ids": [dest_id]}
                },
                "passengers": {
                    "adults": adults,
                    "children": children,
                    "infants": infants
                }
            },
            "filter": {
                "transportTypes": ["FLIGHT"],
                "limit": limit
            },
            "options": {
                "currency": currency,
                "locale": locale,
                "market": locale,
                "partner": "skypicker"
            }
        }
    }
    
    # Dátumok hozzáadása
    if date_from or date_to:
        departure_date = {}
        if date_from:
            departure_date["start"] = f"{date_from}T00:00:00"
        if date_to:
            departure_date["end"] = f"{date_to}T23:59:59"
        graphql_payload["variables"]["search"]["itinerary"]["outboundDepartureDate"] = departure_date
    
    # Átszállások szűréser
    if direct_flights_only:
        graphql_payload["variables"]["filter"]["maxStopovers"] = 0
    elif max_stopovers is not None:
        graphql_payload["variables"]["filter"]["maxStopovers"] = max_stopovers
    
    # API hívás
    headers = {
        "Content-Type": "application/json",
        "kw-umbrella-token": tokens["umbrella_token"],
        "kw-skypicker-visitor-uniqid": tokens["visitor_id"],
        "kw-x-rand-id": tokens["rand_id"],
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    response = requests.post(
        "https://api.skypicker.com/umbrella/v2/graphql?featureName=SearchOneWayItinerariesQuery",
        json=graphql_payload,
        headers=headers,
        timeout=30
    )
    
    data = response.json()
    
    # Hibakezelés
    if "errors" in data:
        print("❌ GraphQL hibák:")
        for error in data["errors"]:
            print(f"  - {error['message']}")
        return pd.DataFrame()
    
    if not data.get("data") or not data["data"].get("onewayItineraries"):
        print("❌ Nem érkezett adat")
        return pd.DataFrame()
    
    result = data["data"]["onewayItineraries"]
    
    if result["__typename"] != "Itineraries":
        print(f"❌ Hiba: {result.get('error', 'Ismeretlen hiba')}")
        return pd.DataFrame()
    
    itineraries = result.get("itineraries", [])
    print(f"✅ {len(itineraries)} járat\n")
    
    # DataFrame építése
    flights_data = []
    
    for flight in itineraries:
        if flight["__typename"] != "ItineraryOneWay":
            continue
        
        # Alapadatok
        price = float(flight["price"]["amount"])
        price_eur = float(flight["priceEur"]["amount"])
        
        # Sector (útvonal)
        sector = flight["sector"]
        segments = sector["sectorSegments"]
        first_seg = segments[0]["segment"]
        last_seg = segments[-1]["segment"]
        
        dep_airport = first_seg["source"]["station"]["code"]
        dep_city = first_seg["source"]["station"]["city"]["name"]
        dep_time = first_seg["source"]["localTime"]
        dep_utc = first_seg["source"]["utcTimeIso"]
        
        arr_airport = last_seg["destination"]["station"]["code"]
        arr_city = last_seg["destination"]["station"]["city"]["name"]
        arr_time = last_seg["destination"]["localTime"]
        arr_utc = last_seg["destination"]["utcTimeIso"]
        
        duration_hours = sector["duration"] / 3600
        stops = len(segments) - 1
        
        # Légitársaságok
        carriers = list(set([seg["segment"]["carrier"]["name"] for seg in segments]))
        carrier_codes = list(set([seg["segment"]["carrier"]["code"] for seg in segments]))
        
        # Booking URL
        booking_url = None
        if flight.get("bookingOptions") and flight["bookingOptions"]["edges"]:
            booking_url = flight["bookingOptions"]["edges"][0]["node"].get("bookingUrl")
        
        flights_data.append({
            "id": flight["id"],
            "origin": origin,
            "destination": destination,
            "dep_city": dep_city,
            "dep_airport": dep_airport,
            "dep_time": dep_time,
            "dep_utc": dep_utc,
            "arr_city": arr_city,
            "arr_airport": arr_airport,
            "arr_time": arr_time,
            "arr_utc": arr_utc,
            "duration_h": round(duration_hours, 1),
            "stops": stops,
            "carriers": ", ".join(carriers),
            "carrier_codes": ", ".join(carrier_codes),
            "price_huf": price,
            "price_eur": price_eur,
            "provider": flight["provider"]["name"],
            "booking_url": booking_url
        })
    
    df = pd.DataFrame(flights_data)
    
    if not df.empty:
        # Dátum oszlopok konvertálása
        df["dep_time"] = pd.to_datetime(df["dep_time"])
        df["arr_time"] = pd.to_datetime(df["arr_time"])
        df["dep_utc"] = pd.to_datetime(df["dep_utc"])
        df["arr_utc"] = pd.to_datetime(df["arr_utc"])
        
        # Rendezés ár szerint
        df = df.sort_values("price_huf").reset_index(drop=True)
    
    return df


def create_return_combinations(
    outbound_df: pd.DataFrame,
    inbound_df: pd.DataFrame,
    min_stay_days: int = 1,
    max_stay_days: Optional[int] = None
) -> pd.DataFrame:
    """
    Oda-vissza kombinációk generálása két egyirányú DataFrame-ből.
    
    Args:
        outbound_df: Oda járatok DataFrame
        inbound_df: Vissza járatok DataFrame
        min_stay_days: Min tartózkodás napokban
        max_stay_days: Max tartózkodás napokban (None = korlátlan)
        
    Returns:
        DataFrame a visszajárat kombinációkkal
    """
    print(f"\n🔄 Kombinációk generálása...")
    print(f"   Oda járatok: {len(outbound_df)}")
    print(f"   Vissza járatok: {len(inbound_df)}")
    
    combinations = []
    
    for _, out_flight in outbound_df.iterrows():
        for _, in_flight in inbound_df.iterrows():
            # Ellenőrzés: visszaút később van-e mint odaút
            stay_days = (in_flight["dep_time"] - out_flight["arr_time"]).days
            
            if stay_days < min_stay_days:
                continue
            
            if max_stay_days and stay_days > max_stay_days:
                continue
            
            total_price_huf = out_flight["price_huf"] + in_flight["price_huf"]
            total_price_eur = out_flight["price_eur"] + in_flight["price_eur"]
            
            combinations.append({
                # Outbound
                "out_id": out_flight["id"],
                "out_dep_city": out_flight["dep_city"],
                "out_dep_airport": out_flight["dep_airport"],
                "out_dep_time": out_flight["dep_time"],
                "out_arr_city": out_flight["arr_city"],
                "out_arr_airport": out_flight["arr_airport"],
                "out_arr_time": out_flight["arr_time"],
                "out_duration_h": out_flight["duration_h"],
                "out_stops": out_flight["stops"],
                "out_carriers": out_flight["carriers"],
                "out_price_huf": out_flight["price_huf"],
                "out_booking_url": out_flight["booking_url"],
                
                # Inbound
                "in_id": in_flight["id"],
                "in_dep_time": in_flight["dep_time"],
                "in_arr_time": in_flight["arr_time"],
                "in_duration_h": in_flight["duration_h"],
                "in_stops": in_flight["stops"],
                "in_carriers": in_flight["carriers"],
                "in_price_huf": in_flight["price_huf"],
                "in_booking_url": in_flight["booking_url"],
                
                # Összesített
                "stay_days": stay_days,
                "total_price_huf": total_price_huf,
                "total_price_eur": total_price_eur,
                "total_stops": out_flight["stops"] + in_flight["stops"]
            })
    
    df = pd.DataFrame(combinations)
    
    if not df.empty:
        df = df.sort_values("total_price_huf").reset_index(drop=True)
        print(f"✅ {len(df)} érvényes kombináció\n")
    else:
        print("❌ Nincs érvényes kombináció\n")
    
    return df


# ===== PÉLDA HASZNÁLAT =====

if __name__ == "__main__":
    # 1. Tokenek megszerzése (csak egyszer kell)
    tokens = get_kiwi_tokens(headless=False)
    
    # 2. Egyirányú járatok lekérése MINDKÉT IRÁNYBAN
    
    # Oda: Budapest -> Barcelona (pl. január 20-30 között)
    outbound_flights = search_one_way_flights(
        origin="budapest_hu",
        destination="barcelona_es",
        tokens=tokens,
        date_from="2026-01-20",
        date_to="2026-01-30",
        adults=1,
        limit=100,
        direct_flights_only=False
    )
    
    # Vissza: Barcelona -> Budapest (pl. február 1-15 között)
    inbound_flights = search_one_way_flights(
        origin="barcelona_es",
        destination="budapest_hu",
        tokens=tokens,
        date_from="2026-02-01",
        date_to="2026-02-15",
        adults=1,
        limit=100,
        direct_flights_only=False
    )
    
    # 3. Kombinációk készítése
    if not outbound_flights.empty and not inbound_flights.empty:
        return_trips = create_return_combinations(
            outbound_flights,
            inbound_flights,
            min_stay_days=2,  # Min 2 nap tartózkodás
            max_stay_days=14  # Max 14 nap
        )
        
        # 4. Top 10 legolcsóbb
        if not return_trips.empty:
            print("📊 TOP 10 LEGOLCSÓBB ODA-VISSZA JÁRAT:\n")
            
            for i, trip in return_trips.head(10).iterrows():
                print(f"{i+1}. {trip['total_price_huf']:,.0f} HUF ({trip['stay_days']} éj)")
                print(f"   ODA: {trip['out_dep_time'].strftime('%m-%d %H:%M')} → {trip['out_arr_time'].strftime('%m-%d %H:%M')} "
                      f"({trip['out_duration_h']}h, {trip['out_stops']} stop)")
                print(f"        {trip['out_carriers']}")
                print(f"   VISSZA: {trip['in_dep_time'].strftime('%m-%d %H:%M')} → {trip['in_arr_time'].strftime('%m-%d %H:%M')} "
                      f"({trip['in_duration_h']}h, {trip['in_stops']} stop)")
                print(f"           {trip['in_carriers']}")
                print()

🚀 Tokenek megszerzése...
⏳ Várakozás a GraphQL hívásokra...
✅ Tokenek megszerzve

🔍 Egyirányú járatok: budapest_hu → barcelona_es (2026-01-20 - 2026-01-30)
✅ 50 járat

🔍 Egyirányú járatok: barcelona_es → budapest_hu (2026-02-01 - 2026-02-15)
✅ 50 járat


🔄 Kombinációk generálása...
   Oda járatok: 50
   Vissza járatok: 50
✅ 1487 érvényes kombináció

📊 TOP 10 LEGOLCSÓBB ODA-VISSZA JÁRAT:

1. 20,901 HUF (8 éj)
   ODA: 01-27 15:40 → 01-27 18:20 (2.7h, 0 stop)
        Ryanair
   VISSZA: 02-04 18:50 → 02-04 21:30 (2.7h, 0 stop)
           Ryanair

2. 22,457 HUF (13 éj)
   ODA: 01-27 15:40 → 01-27 18:20 (2.7h, 0 stop)
        Ryanair
   VISSZA: 02-10 12:35 → 02-10 15:15 (2.7h, 0 stop)
           Ryanair

3. 22,457 HUF (6 éj)
   ODA: 01-27 15:40 → 01-27 18:20 (2.7h, 0 stop)
        Ryanair
   VISSZA: 02-03 12:35 → 02-03 15:15 (2.7h, 0 stop)
           Ryanair

4. 22,457 HUF (12 éj)
   ODA: 01-27 15:40 → 01-27 18:20 (2.7h, 0 stop)
        Ryanair
   VISSZA: 02-09 06:25 → 02-09 09:05 (2.7h, 0 s

In [19]:
# AirHint

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time

# Böngésző inicializálás
driver = webdriver.Chrome()
driver.get("https://www.airhint.com")

wait = WebDriverWait(driver, 10)

try:
    # Cookie gomb klikk
    cookie_button = wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/div/div/div/div[2]/div/button[2]")))
    cookie_button.click()

    # Origin reptér kiválasztása
    origin_container = wait.until(EC.element_to_be_clickable((By.ID, "select2-origin-container")))
    origin_container.click()
    
    # Várakozás a dropdown megjelenésére és keresőmező
    search_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".select2-search__field")))
    search_input.send_keys("BUD")
    time.sleep(0.5)
    search_input.send_keys(Keys.ENTER)
    
    # Első találat kiválasztása
    first_result = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".select2-results__option--highlighted")))
    first_result.click()
    
    # Destination reptér kiválasztása
    dest_container = wait.until(EC.element_to_be_clickable((By.ID, "select2-destination-container")))
    dest_container.click()

    wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, ".select2-dropdown")
    ))

    search_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".select2-search__field")))
    search_input.clear()
    search_input.send_keys("BCN")
    time.sleep(0.5)

    search_input.send_keys(Keys.ENTER)

    
    first_result = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".select2-results__option--highlighted")))
    first_result.click()
    
    # Departure dátum kiválasztása
    departure_input = wait.until(EC.element_to_be_clickable((By.ID, "departure")))
    departure_input.click()
    
    # Február 17-re kattintás a datepickerben
    date_feb_17 = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "td.active.day[data-date='1771286400000']")))
    date_feb_17.click()
    
    # Return dátum kiválasztása
    return_input = wait.until(EC.element_to_be_clickable((By.ID, "return_date")))
    return_input.click()
    time.sleep(0.3)
    
    # Egy későbbi dátum kiválasztása (pl. Február 24)
    date_feb_24 = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "td.day[data-date='1771891200000']")))
    date_feb_24.click()
    
    # Accommodation checkbox kikattintása
    accommodation_checkbox = driver.find_element(By.ID, "findAccomodation")
    if accommodation_checkbox.is_selected():
        accommodation_checkbox.click()
    
    # Find flight gomb megnyomása
    find_button = wait.until(EC.element_to_be_clickable((By.ID, "find_btn")))
    find_button.click()
    
    print("Keresés elindítva!")
    time.sleep(5)  # Várunk, hogy lássuk az eredményt

except Exception as e:
    print(f"Hiba történt: {e}")

finally:
    driver.quit()

Hiba történt: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff67a8c88e5
	0x7ff67a8c8940
	0x7ff67a6a165d
	0x7ff67a6f9a33
	0x7ff67a6f9d3c
	0x7ff67a74df67
	0x7ff67a74ac97
	0x7ff67a6eac29
	0x7ff67a6eba93
	0x7ff67abe0640
	0x7ff67abdaf80
	0x7ff67abf96e6
	0x7ff67a8e5de4
	0x7ff67a8eed8c
	0x7ff67a8d2004
	0x7ff67a8d21b5
	0x7ff67a8b7ee2
	0x7ffdb1f47374
	0x7ffdb25fcc91



In [21]:
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict

def collect_historical_data(
    origin: str,
    destination: str,
    tokens: dict,
    days_ahead: int = 90,
    monitoring_period: int = 30
) -> pd.DataFrame:
    """
    Historikus áradatok gyűjtése adott időszakra.
    
    Args:
        origin: Indulási hely
        destination: Célállomás
        tokens: API tokenek
        days_ahead: Hány napra előre nézzünk járatokat
        monitoring_period: Hány napon keresztül monitorozunk
        
    Returns:
        DataFrame historikus árakkal és metaadatokkal
    """
    print(f"📊 Historikus adatgyűjtés: {origin} → {destination}")
    print(f"   Időablak: {days_ahead} nap előre, {monitoring_period} napos monitoring\n")
    
    all_snapshots = []
    
    for day_offset in range(monitoring_period):
        snapshot_date = datetime.now().date()
        date_from = (datetime.now() + timedelta(days=1)).date()
        date_to = (datetime.now() + timedelta(days=days_ahead)).date()
        
        print(f"📸 Snapshot {day_offset + 1}/{monitoring_period} - {snapshot_date}")
        
        flights = search_one_way_flights(
            origin=origin,
            destination=destination,
            tokens=tokens,
            date_from=str(date_from),
            date_to=str(date_to),
            limit=200
        )
        
        if not flights.empty:
            flights['snapshot_date'] = snapshot_date
            flights['days_until_departure'] = (flights['dep_time'].dt.date - snapshot_date).dt.days
            all_snapshots.append(flights)
        
        # Szimulált időutazás - valós esetben napi futtatás
        # time.sleep(3600 * 24)  # 24 óra várakozás
    
    if all_snapshots:
        historical_df = pd.concat(all_snapshots, ignore_index=True)
        historical_df.to_csv(f"historical_{origin}_{destination}.csv", index=False)
        print(f"\n✅ {len(historical_df)} historikus rekord mentve\n")
        return historical_df
    
    return pd.DataFrame()

def calculate_price_statistics(
    historical_df: pd.DataFrame,
    departure_date: str
) -> dict:
    """
    Árstatisztikák számítása egy adott járatra.
    
    Args:
        historical_df: Historikus adatok
        departure_date: Indulási dátum (YYYY-MM-DD)
        
    Returns:
        Statisztikák dictionary
    """
    dep_date = pd.to_datetime(departure_date).date()
    
    # Szűrés az adott indulási dátumra
    flight_history = historical_df[
        historical_df['dep_time'].dt.date == dep_date
    ].copy()
    
    if flight_history.empty:
        return None
    
    prices = flight_history['price_huf'].values
    
    stats = {
        'mean': np.mean(prices),
        'median': np.median(prices),
        'std': np.std(prices),
        'min': np.min(prices),
        'max': np.max(prices),
        'p10': np.percentile(prices, 10),
        'p25': np.percentile(prices, 25),
        'p50': np.percentile(prices, 50),
        'p75': np.percentile(prices, 75),
        'p90': np.percentile(prices, 90),
        'sample_size': len(prices),
        'price_volatility': np.std(prices) / np.mean(prices) if np.mean(prices) > 0 else 0
    }
    
    return stats

def validate_ticket(
    price: float,
    departure_date: str,
    historical_df: pd.DataFrame,
    include_prediction: bool = True
) -> dict:
    """
    Jegy validálás: objektív értékelés egy adott árról.
    
    Args:
        price: Vizsgált ár (HUF)
        departure_date: Indulási dátum
        historical_df: Historikus adatok
        include_prediction: Predikció készítése
        
    Returns:
        Validációs eredmény
    """
    print(f"🔍 Jegy validálás: {price:,.0f} HUF")
    
    stats = calculate_price_statistics(historical_df, departure_date)
    
    if not stats:
        return {
            'status': 'insufficient_data',
            'message': 'Nincs elég historikus adat az értékeléshez'
        }
    
    # Percentilis számítás
    percentile = (np.sum(stats['min'] <= price) / stats['sample_size']) * 100 \
                 if stats['sample_size'] > 0 else 50
    
    # Ár kategorizálás
    if percentile <= 15:
        category = 'excellent'
        emoji = '🟢'
    elif percentile <= 35:
        category = 'good'
        emoji = '🟡'
    elif percentile <= 65:
        category = 'fair'
        emoji = '🟠'
    else:
        category = 'expensive'
        emoji = '🔴'
    
    # Döntési ajánlás
    if percentile <= 25:
        recommendation = 'BUY_NOW'
        confidence = 0.85
        message = 'Kivételesen jó ár! Vedd meg most.'
    elif percentile <= 50:
        recommendation = 'BUY'
        confidence = 0.70
        message = 'Jó ár, érdemes megvenni.'
    elif percentile <= 75:
        recommendation = 'CONSIDER'
        confidence = 0.55
        message = 'Átlagos ár. Érdemes még figyelni.'
    else:
        recommendation = 'WAIT'
        confidence = 0.65
        message = 'Drága. Várj alacsonyabb árat.'
    
    # Predikció (egyszerű trend-alapú)
    prediction = None
    if include_prediction:
        prediction = predict_price_trend(historical_df, departure_date)
    
    result = {
        'status': 'validated',
        'price': price,
        'category': category,
        'emoji': emoji,
        'percentile': round(percentile, 1),
        'statistics': stats,
        'recommendation': recommendation,
        'confidence': confidence,
        'message': message,
        'prediction': prediction
    }
    
    return result

def predict_price_trend(
    historical_df: pd.DataFrame,
    departure_date: str,
    horizon_days: int = 7
) -> dict:
    """
    Ártrend predikció egyszerű statisztikai módszerrel.
    
    Args:
        historical_df: Historikus adatok
        departure_date: Indulási dátum
        horizon_days: Előrejelzési horizont napokban
        
    Returns:
        Predikciós eredmény
    """
    dep_date = pd.to_datetime(departure_date).date()
    
    # Szűrés az adott járatra
    flight_history = historical_df[
        historical_df['dep_time'].dt.date == dep_date
    ].copy()
    
    if len(flight_history) < 3:
        return {'status': 'insufficient_data'}
    
    # Idősor rendezése
    flight_history = flight_history.sort_values('snapshot_date')
    
    # Utolsó 7 nap átlagos ára
    recent_prices = flight_history.tail(7)['price_huf'].values
    older_prices = flight_history.head(max(1, len(flight_history) - 7))['price_huf'].values
    
    recent_mean = np.mean(recent_prices)
    older_mean = np.mean(older_prices) if len(older_prices) > 0 else recent_mean
    
    # Trend számítás
    price_change = recent_mean - older_mean
    change_pct = (price_change / older_mean * 100) if older_mean > 0 else 0
    
    # Volatilitás
    volatility = np.std(recent_prices) / recent_mean if recent_mean > 0 else 0
    
    # Predikció
    if change_pct < -5:
        trend = 'FALLING'
        emoji = '📉'
        probability_cheaper = 0.65
    elif change_pct > 5:
        trend = 'RISING'
        emoji = '📈'
        probability_cheaper = 0.25
    else:
        trend = 'STABLE'
        emoji = '➡️'
        probability_cheaper = 0.45
    
    # Worst-case scenario
    worst_case_increase = recent_mean * (1 + volatility * 2)
    
    return {
        'status': 'predicted',
        'trend': trend,
        'emoji': emoji,
        'change_pct': round(change_pct, 1),
        'probability_cheaper_soon': probability_cheaper,
        'current_avg': round(recent_mean, 0),
        'volatility': round(volatility, 3),
        'worst_case_price': round(worst_case_increase, 0),
        'horizon_days': horizon_days
    }

def find_optimal_dates(
    origin: str,
    destination: str,
    tokens: dict,
    date_window_start: str,
    date_window_end: str,
    flexibility_days: int = 3,
    preference_weights: dict = None
) -> pd.DataFrame:
    """
    Optimális dátumok keresése rugalmas időablakban.
    
    Args:
        origin: Indulási hely
        destination: Célállomás
        tokens: API tokenek
        date_window_start: Ablak kezdete (YYYY-MM-DD)
        date_window_end: Ablak vége (YYYY-MM-DD)
        flexibility_days: +/- napok rugalmassága
        preference_weights: Súlyok (price, time, comfort)
        
    Returns:
        Top opciók DataFrame-je optimalizációs score-ral
    """
    if preference_weights is None:
        preference_weights = {'price': 0.6, 'time': 0.2, 'comfort': 0.2}
    
    print(f"🎯 Optimális dátumok keresése:")
    print(f"   Ablak: {date_window_start} - {date_window_end}")
    print(f"   Rugalmasság: ±{flexibility_days} nap\n")
    
    # Járatok lekérése teljes ablakra
    flights = search_one_way_flights(
        origin=origin,
        destination=destination,
        tokens=tokens,
        date_from=date_window_start,
        date_to=date_window_end,
        limit=300
    )
    
    if flights.empty:
        return pd.DataFrame()
    
    # Normalizálás 0-1 skálára
    flights['price_norm'] = 1 - (flights['price_huf'] - flights['price_huf'].min()) / \
                                 (flights['price_huf'].max() - flights['price_huf'].min())
    
    flights['time_norm'] = 1 - (flights['duration_h'] - flights['duration_h'].min()) / \
                                (flights['duration_h'].max() - flights['duration_h'].min())
    
    # Komfort score: direkt járat jobb, kevesebb stop jobb
    flights['comfort_score'] = 1 - (flights['stops'] / flights['stops'].max()) \
                               if flights['stops'].max() > 0 else 1
    
    # Összesített optimalizációs score
    flights['optimization_score'] = (
        preference_weights['price'] * flights['price_norm'] +
        preference_weights['time'] * flights['time_norm'] +
        preference_weights['comfort'] * flights['comfort_score']
    ) * 100
    
    # Csoportosítás dátum szerint - best per day
    best_per_day = flights.loc[
        flights.groupby(flights['dep_time'].dt.date)['optimization_score'].idxmax()
    ].copy()
    
    best_per_day['savings_vs_avg'] = (
        (flights['price_huf'].mean() - best_per_day['price_huf']) / 
        flights['price_huf'].mean() * 100
    )
    
    result = best_per_day.sort_values('optimization_score', ascending=False).head(10)
    
    print(f"✅ Top 10 optimális dátum találva\n")
    
    return result

def interactive_preference_elicitation() -> dict:
    """
    Interaktív preferencia-feltárás páronkénti összehasonlítással.
    
    Returns:
        Súlyok dictionary
    """
    print("🎯 PREFERENCIA MEGHATÁROZÁS\n")
    print("Válaszd ki, melyik bosszant jobban:\n")
    
    comparisons = [
        {
            'question': '1. Mi bosszant jobban?',
            'option_a': '+15,000 HUF drágább jegy',
            'option_b': '+2 óra hosszabb út',
            'weights_if_a': {'price': 0.3, 'time': 0.7},
            'weights_if_b': {'price': 0.7, 'time': 0.3}
        },
        {
            'question': '2. Mi bosszant jobban?',
            'option_a': '1 átszállás',
            'option_b': '+8,000 HUF',
            'weights_if_a': {'comfort': 0.3, 'price': 0.7},
            'weights_if_b': {'comfort': 0.7, 'price': 0.3}
        },
        {
            'question': '3. Mi bosszant jobban?',
            'option_a': 'Reggel 6:00-kor indulás',
            'option_b': '+1.5 óra utazási idő',
            'weights_if_a': {'comfort': 0.3, 'time': 0.7},
            'weights_if_b': {'comfort': 0.7, 'time': 0.3}
        }
    ]
    
    aggregated_weights = {'price': 0, 'time': 0, 'comfort': 0}
    
    # Szimuláció - valós esetben input()
    print("(Szimuláció: válaszok automatikusan generálva)")
    
    for comp in comparisons:
        print(f"\n{comp['question']}")
        print(f"  A) {comp['option_a']}")
        print(f"  B) {comp['option_b']}")
        
        # Szimulált válasz
        choice = np.random.choice(['a', 'b'])
        print(f"  Válasz: {choice.upper()}")
        
        weights = comp['weights_if_a'] if choice == 'a' else comp['weights_if_b']
        
        for key, value in weights.items():
            aggregated_weights[key] += value
    
    # Normalizálás
    total = sum(aggregated_weights.values())
    final_weights = {k: v / total for k, v in aggregated_weights.items()}
    
    print(f"\n✅ Számított preferenciák:")
    for key, value in final_weights.items():
        print(f"   {key.capitalize()}: {value:.1%}")
    
    return final_weights

class PriceAlertMonitor:
    """Folyamatos árfigyelő rendszer."""
    
    def __init__(self, origin: str, destination: str, tokens: dict):
        self.origin = origin
        self.destination = destination
        self.tokens = tokens
        self.alerts = []
        self.baseline_price = None
    
    def set_alert(
        self,
        departure_date: str,
        threshold_type: str = 'absolute',  # 'absolute' vagy 'percentage'
        threshold_value: float = None,
        alert_condition: str = 'below'  # 'below' vagy 'drop'
    ):
        """
        Árfigyel beállítása.
        
        Args:
            departure_date: Indulási dátum
            threshold_type: 'absolute' (fix ár) vagy 'percentage' (% változás)
            threshold_value: Küszöbérték
            alert_condition: 'below' (ár alatt) vagy 'drop' (X%-os esés)
        """
        alert = {
            'id': len(self.alerts) + 1,
            'departure_date': departure_date,
            'threshold_type': threshold_type,
            'threshold_value': threshold_value,
            'alert_condition': alert_condition,
            'created_at': datetime.now(),
            'triggered': False
        }
        
        self.alerts.append(alert)
        print(f"✅ Alert #{alert['id']} beállítva: {departure_date}")
    
    def check_alerts(self) -> list:
        """
        Figyelmeztetések ellenőrzése.
        
        Returns:
            Lista a kiváltott alertekkel
        """
        print(f"\n🔔 Alertek ellenőrzése ({len(self.alerts)} aktív)...\n")
        
        triggered_alerts = []
        
        for alert in self.alerts:
            if alert['triggered']:
                continue
            
            # Aktuális árak lekérése
            flights = search_one_way_flights(
                origin=self.origin,
                destination=self.destination,
                tokens=self.tokens,
                date_from=alert['departure_date'],
                date_to=alert['departure_date'],
                limit=50
            )
            
            if flights.empty:
                continue
            
            current_best_price = flights['price_huf'].min()
            
            # Alert logika
            should_trigger = False
            
            if alert['alert_condition'] == 'below':
                if current_best_price <= alert['threshold_value']:
                    should_trigger = True
            
            elif alert['alert_condition'] == 'drop':
                if self.baseline_price:
                    drop_pct = (self.baseline_price - current_best_price) / self.baseline_price * 100
                    if drop_pct >= alert['threshold_value']:
                        should_trigger = True
            
            if should_trigger:
                alert['triggered'] = True
                alert['trigger_price'] = current_best_price
                alert['trigger_time'] = datetime.now()
                triggered_alerts.append(alert)
                
                print(f"🚨 ALERT #{alert['id']} KIVÁLTVA!")
                print(f"   Ár: {current_best_price:,.0f} HUF")
                print(f"   Dátum: {alert['departure_date']}\n")
        
        return triggered_alerts
    
if __name__ == "__main__":
    
    # ===== INICIALIZÁLÁS =====
    print("=" * 60)
    print("✈️  FLIGHT DECISION INTELLIGENCE SYSTEM")
    print("=" * 60 + "\n")
    
    tokens = get_kiwi_tokens(headless=True)
    
    origin = "budapest_hu"
    destination = "barcelona_es"
    
    # ===== USE CASE 1: TICKET VALIDATION =====
    print("\n" + "=" * 60)
    print("🔍 USE CASE 1: TICKET VALIDATION")
    print("=" * 60 + "\n")
    
    # Szimuláljuk, hogy van historikus adatunk
    print("📊 Historikus adatok betöltése...")
    # historical_df = collect_historical_data(origin, destination, tokens)
    
    # Demo célra generálunk fake historikus adatot
    np.random.seed(42)
    fake_history = []
    base_date = datetime.now() + timedelta(days=30)
    
    for _ in range(100):
        fake_history.append({
            'dep_time': base_date,
            'price_huf': np.random.normal(45000, 8000),
            'snapshot_date': datetime.now().date()
        })
    
    historical_df = pd.DataFrame(fake_history)
    historical_df['dep_time'] = pd.to_datetime(historical_df['dep_time'])
    print("✅ 100 historikus rekord betöltve (demo)\n")
    
    # Validálás
    test_price = 38000
    validation = validate_ticket(
        price=test_price,
        departure_date=str(base_date.date()),
        historical_df=historical_df
    )
    
    print(f"\n{validation['emoji']} ÁR ÉRTÉKELÉS:")
    print(f"   Kategória: {validation['category']}")
    print(f"   Percentilis: P{validation['percentile']}")
    print(f"   Ajánlás: {validation['recommendation']}")
    print(f"   Üzenet: {validation['message']}")
    print(f"   Magabiztosság: {validation['confidence']:.0%}")
    
    if validation['prediction']:
        pred = validation['prediction']
        print(f"\n📊 PREDIKCIÓ ({pred['horizon_days']} napra):")
        print(f"   Trend: {pred['emoji']} {pred['trend']}")
        print(f"   Valószínűség olcsóbbra: {pred['probability_cheaper_soon']:.0%}")
        print(f"   Worst-case ár: {pred['worst_case_price']:,.0f} HUF")
    
    # ===== USE CASE 2: FLEXIBLE DATE OPTIMIZATION =====
    print("\n" + "=" * 60)
    print("🎯 USE CASE 2: FLEXIBLE DATE OPTIMIZATION")
    print("=" * 60 + "\n")
    
    optimal_dates = find_optimal_dates(
        origin=origin,
        destination=destination,
        tokens=tokens,
        date_window_start="2026-01-20",
        date_window_end="2026-01-30",
        flexibility_days=3,
        preference_weights={'price': 0.5, 'time': 0.3, 'comfort': 0.2}
    )
    
    if not optimal_dates.empty:
        print("📊 TOP 5 OPTIMÁLIS DÁTUM:\n")
        for i, row in optimal_dates.head(5).iterrows():
            print(f"{i+1}. {row['dep_time'].strftime('%Y-%m-%d')} - Score: {row['optimization_score']:.1f}")
            print(f"   Ár: {row['price_huf']:,.0f} HUF ({row['savings_vs_avg']:+.1f}% vs átlag)")
            print(f"   Út: {row['duration_h']}h, {row['stops']} stop")
            print(f"   Légitársaság: {row['carriers']}\n")
    
    # ===== USE CASE 3: INTERACTIVE PREFERENCE =====
    print("\n" + "=" * 60)
    print("🎯 USE CASE 3: PREFERENCE ELICITATION")
    print("=" * 60 + "\n")
    
    user_preferences = interactive_preference_elicitation()
    
    # ===== USE CASE 4: PRICE ALERTS =====
    print("\n" + "=" * 60)
    print("🔔 USE CASE 4: PRICE ALERT SYSTEM")
    print("=" * 60 + "\n")
    
    monitor = PriceAlertMonitor(origin, destination, tokens)
    
    # Alert beállítások
    monitor.set_alert(
        departure_date="2026-01-25",
        threshold_type='absolute',
        threshold_value=40000,
        alert_condition='below'
    )
    
    monitor.set_alert(
        departure_date="2026-01-26",
        threshold_type='percentage',
        threshold_value=10,  # 10% esés
        alert_condition='drop'
    )
    
    # Ellenőrzés (valós esetben cron job vagy background task)
    triggered = monitor.check_alerts()
    
    print("\n" + "=" * 60)
    print("✅ WORKFLOW BEFEJEZVE")
    print("=" * 60 + "\n")

✈️  FLIGHT DECISION INTELLIGENCE SYSTEM

🚀 Tokenek megszerzése...
⏳ Várakozás a GraphQL hívásokra...
✅ Tokenek megszerzve


🔍 USE CASE 1: TICKET VALIDATION

📊 Historikus adatok betöltése...
✅ 100 historikus rekord betöltve (demo)

🔍 Jegy validálás: 38,000 HUF

🟢 ÁR ÉRTÉKELÉS:
   Kategória: excellent
   Percentilis: P1.0
   Ajánlás: BUY_NOW
   Üzenet: Kivételesen jó ár! Vedd meg most.
   Magabiztosság: 85%

📊 PREDIKCIÓ (7 napra):
   Trend: ➡️ STABLE
   Valószínűség olcsóbbra: 45%
   Worst-case ár: 50,074 HUF

🎯 USE CASE 2: FLEXIBLE DATE OPTIMIZATION

🎯 Optimális dátumok keresése:
   Ablak: 2026-01-20 - 2026-01-30
   Rugalmasság: ±3 nap

🔍 Egyirányú járatok: budapest_hu → barcelona_es (2026-01-20 - 2026-01-30)
✅ 50 járat

✅ Top 10 optimális dátum találva

📊 TOP 5 OPTIMÁLIS DÁTUM:

1. 2026-01-27 - Score: 100.0
   Ár: 11,956 HUF (+44.3% vs átlag)
   Út: 2.7h, 0 stop
   Légitársaság: Ryanair

2. 2026-01-22 - Score: 95.4
   Ár: 13,975 HUF (+34.8% vs átlag)
   Út: 2.7h, 0 stop
   Légitársaság